## Building the Transformer Encoder

###  **Objective**

Implement the core components of a Transformer Encoder block: Scaled Dot-Product Attention (SDPA) and Multi-Head Attention (MHA). Then, combine them with a Position-wise Feed-Forward Network (FFN) and residual connections to build a complete `EncoderLayer` module.

###  **Assignment Structure**

This assignment will contain:

1.  **Setup Code:** Imports and configuration variables.
2.  **Part 1: SDPA:** A skeleton function for the student to complete, followed by verification code and questions.
3.  **Part 2: MHA:** A skeleton class for the student to complete, followed by verification code and questions.
4.  **Part 3: FFN:** A skeleton class for the student to complete, followed by verification code and questions.
5.  **Part 4: EncoderLayer:** A skeleton class for the student to complete, followed by verification code and questions.

In [ ]:
import torch
import torch.nn as nn
import math
from torch.nn import functional as F

# --- Configuration for testing ---
# Use these global variables in your verification steps
BATCH_SIZE = 8
SEQ_LENGTH = 12
D_MODEL = 512
NUM_HEADS = 8
D_FF = 2048  # Dimension for the Feed-Forward Network
DROPOUT = 0.1
HEAD_DIM = D_MODEL // NUM_HEADS

# --- Reproducibility ---
# Set the seed for all random operations
torch.manual_seed(42)

print("="*50)
print("RUNNING TRANSFORMER ENCODER")
print(f"PyTorch Version: {torch.__version__}")
print(f"Using Config: D_MODEL={D_MODEL}, NUM_HEADS={NUM_HEADS}")
print("="*50)

RUNNING TRANSFORMER ENCODER
PyTorch Version: 2.10.0+cpu
Using Config: D_MODEL=512, NUM_HEADS=8


-----

### **Part 1: Scaled Dot-Product Attention**

In [ ]:
print("\n--- Part 1: Scaled Dot-Product Attention ---")

def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Calculates the Scaled Dot-Product Attention.

    Args:
        query (torch.Tensor): Shape (batch_size, num_heads, seq_len_q, head_dim)
        key (torch.Tensor): Shape (batch_size, num_heads, seq_len_k, head_dim)
        value (torch.Tensor): Shape (batch_size, num_heads, seq_len_v, head_dim)
        mask (torch.Tensor, optional): Shape (batch_size, 1, 1, seq_len_k) or
                                       (batch_size, 1, seq_len_q, seq_len_k)

    Returns:
        torch.Tensor: Output tensor, shape (batch_size, num_heads, seq_len_q, head_dim)
        torch.Tensor: Attention weights, shape (batch_size, num_heads, seq_len_q, seq_len_k)
    """
    # 1. Get the dimension of the key (d_k)
    d_k = key.size(-1)

    # 2. Calculate scores (Q * K^T)
    # Expected shape: (batch_size, num_heads, seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(2, 3))

    # 3. Scale the scores
    scaled_scores = scores / (d_k ** 0.5)

    # 4. Apply mask (if provided)
    if mask is not None:
        # Fill with a very small number (-1e9) where mask is 0
        scaled_scores = scaled_scores.masked_fill(mask == 0, -1e9)

    # 5. Apply softmax to get attention weights
    # (Apply along the last dimension)
    attention_weights = F.softmax(scaled_scores, -1)

    # 6. Get the final output (Attention_Weights * V)
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

# --- Verification for Part 1 ---
print("--- Running Verification for Part 1 ---")
try:
    # Create test tensors
    torch.manual_seed(42)
    q, k, v = (torch.randn(BATCH_SIZE, NUM_HEADS, SEQ_LENGTH, HEAD_DIM) for _ in range(3))
    output, attn_weights = scaled_dot_product_attention(q, k, v)

    assert output.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LENGTH, HEAD_DIM)
    print(" (1a) Output shape is correct.")
    assert attn_weights.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LENGTH, SEQ_LENGTH)
    print(" (1b) Attention weights shape is correct.")
    assert torch.allclose(attn_weights.sum(dim=-1), torch.ones(1), atol=1e-5)
    print(" (1c) Attention weights sum to 1.")
    print(" Part 1 seems correct!")
except Exception as e:
    print(f" Part 1 Failed: {e}")


--- Part 1: Scaled Dot-Product Attention ---
--- Running Verification for Part 1 ---
 (1a) Output shape is correct.
 (1b) Attention weights shape is correct.
 (1c) Attention weights sum to 1.
 Part 1 seems correct!


#### **Questions for Part 1**

**Q1: Conceptual Check**
What is the primary purpose of dividing the dot-product scores by the square root of `d_k` (the key dimension)?

  * (A) To speed up the matrix multiplication.
  * (B) To prevent the softmax function from saturating (i.e., having gradients near zero).
  * (C) To reduce the memory usage of the attention weights.
  * (D) To ensure the `query` and `key` matrices have compatible dimensions.



**Q2: Value Check**
Using the `scaled_dot_product_attention` function you just wrote, run the code block below. What is the **mean** of the resulting `output` tensor? *Enter the answer as a float rounded to 2 decimal places (e.g., `0.12`).*

In [ ]:
# Code for Q2
torch.manual_seed(123)
q_test = torch.randn(2, 4, 5, 8) # B, H, S, D_k
k_test = torch.randn(2, 4, 5, 8)
v_test = torch.randn(2, 4, 5, 8)
output_test, _ = scaled_dot_product_attention(q_test, k_test, v_test)
print(f"Q2 Output Mean: {output_test.mean().item()}")

Q2 Output Mean: 0.06083366274833679





**Q3: Masking Logic**
If you create a causal (look-ahead) mask for a sequence of length 5, what will be the value of `attention_weights[0, 0, 1, 3]` (for the 2nd token attending to the 4th token)?

  * (A) 0.0
  * (B) 0.2
  * (C) 0.5
  * (D) 1.0



-----

### **Part 2: Multi-Head Attention Layer**

In [ ]:
print("\n--- Part 2: Multi-Head Attention Layer ---")

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # 1. Define the four linear layers
        # (wq, wk, wv for projections, wo for output)
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1. Pass Q, K, V through linear layers
        # Shape: (batch_size, seq_len, d_model)
        Q = self.wq(query)
        K = self.wk(key)
        V = self.wv(value)

        # 2. Reshape Q, K, V for multi-head processing
        # Change shape from (batch_size, seq_len, d_model) to
        # (batch_size, num_heads, seq_len, head_dim)
        # Hint: Use .view() and .transpose(1, 2)
        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Call scaled_dot_product_attention
        # (This is the function you wrote in Part 1)
        attention_output, _ = scaled_dot_product_attention(Q, K, V, mask)

        # 4. Concatenate heads back
        # Reshape from (batch_size, num_heads, seq_len, head_dim) to
        # (batch_size, seq_len, d_model)
        # Hint: Use .transpose(1, 2), .contiguous(), and .view()
        attention_output = attention_output.transpose(1, 2).contiguous()
        concatenated_output = attention_output.view(batch_size, -1, self.d_model)

        # 5. Pass through final linear layer (wo)
        final_output = self.wo(concatenated_output)

        return final_output

# --- Verification for Part 2 ---
print("--- Running Verification for Part 2 ---")
try:
    torch.manual_seed(42)
    mha_layer = MultiHeadAttention(D_MODEL, NUM_HEADS)
    x_test = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    output = mha_layer(x_test, x_test, x_test)

    assert output.shape == (BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    print(" (2a) Final output shape is correct.")

    # Calculate expected parameters
    # 4 layers (Q, K, V, O), each with (d_model * d_model) weights
    # and (d_model) biases.
    expected_params = 4 * (D_MODEL * D_MODEL + D_MODEL)
    total_params = sum(p.numel() for p in mha_layer.parameters())
    assert total_params == expected_params
    print(f" (2b) Number of parameters is correct ({total_params}).")
    print(" Part 2 seems correct!")
except Exception as e:
    print(f" Part 2 Failed: {e}")


--- Part 2: Multi-Head Attention Layer ---
--- Running Verification for Part 2 ---
 (2a) Final output shape is correct.
 (2b) Number of parameters is correct (1050624).
 Part 2 seems correct!


#### **Questions for Part 2**

**Q4: Architecture Check**
For our configuration (`D_MODEL=512`, `NUM_HEADS=8`), how many *total trainable parameters* are in the `MultiHeadAttention` module you just built?

  * (A) 786,432
  * (B) 1,050,624
  * (C) 1,048,576
  * (D) 1,050,112


**Q5: Value Check**
Using the `mha_layer` initialized in the verification block (with `seed=42`), run it on the `x_test` input. What is the **sum** of the resulting `output` tensor?

In [ ]:
# Code for Q5
# (Assumes mha_layer and x_test are from the verification block)
torch.manual_seed(42)
mha_layer_q5 = MultiHeadAttention(D_MODEL, NUM_HEADS)
x_test_q5 = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
output_q5 = mha_layer_q5(x_test_q5, x_test_q5, x_test_q5)
print(f"Q5 Output Sum: {output_q5.sum().item()}")

Q5 Output Sum: 135.802490234375



-----

### **Part 3: Position-wise Feed-Forward Network (FFN)**

In [ ]:
print("\n--- Part 3: Position-wise Feed-Forward Network ---")

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        # 1. Define the two linear layers and activation
        # Layer 1: d_model -> d_ff
        # Layer 2: d_ff -> d_model
        # Activation: ReLU
        self.fc1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout()
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # Pass x through fc1, relu, dropout, fc2
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- Verification for Part 3 ---
print("--- Running Verification for Part 3 ---")
try:
    torch.manual_seed(42)
    ffn_layer = PositionwiseFeedForward(D_MODEL, D_FF)
    x_test = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    output = ffn_layer(x_test)

    assert output.shape == (BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    print(" (3a) FFN output shape is correct.")
    print(" Part 3 seems correct!")
except Exception as e:
    print(f" Part 3 Failed: {e}")


--- Part 3: Position-wise Feed-Forward Network ---
--- Running Verification for Part 3 ---
 (3a) FFN output shape is correct.
 Part 3 seems correct!


#### **Questions for Part 3**

**Q6: Conceptual Check**
Why is this component called "Position-wise"?

  * (A) It uses positional encodings to modify the weights.
  * (B) It applies the *same* linear transformation to *each position* (token) independently.
  * (C) It shuffles the positions of the tokens before applying the linear layers.
  * (D) It only operates on the first and last position of the sequence.


-----

### **Part 4: Assembling the Full Encoder Layer**

In [ ]:
print("\n--- Part 4: Assembling the Full Encoder Layer ---")

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        # 1. Initialize MHA, FFN, LayerNorms, and Dropouts
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)

        # Two LayerNorms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Two Dropout layers
        self.dropout1 = nn.Dropout()
        self.dropout2 = nn.Dropout()

    def forward(self, x, mask=None):
        # 1. MHA sub-layer (Attention + Add & Norm)
        # 1a. Calculate attention output
        attn_output = self.mha(x, x, x, mask)
        # 1b. Apply dropout, add residual, and normalize (x + dropout(attn_output))
        x = self.norm1(x + self.dropout1(attn_output))

        # 2. FFN sub-layer (FFN + Add & Norm)
        # 2a. Calculate FFN output
        ffn_output = self.ffn(x)
        # 2b. Apply dropout, add residual, and normalize (x + dropout(ffn_output))
        x = self.norm2(x + self.dropout2(ffn_output))

        return x

# --- Verification for Part 4 ---
print("--- Running Verification for Part 4 ---")
try:
    torch.manual_seed(42)
    # Use the global config variables
    encoder_layer = EncoderLayer(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
    x_test = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    output = encoder_layer(x_test, mask=None)

    assert output.shape == (BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    print(" (4a) Final EncoderLayer output shape is correct.")
    print(" Part 4 seems correct! Assignment complete.")
except Exception as e:
    print(f" Part 4 Failed: {e}")


--- Part 4: Assembling the Full Encoder Layer ---
--- Running Verification for Part 4 ---
 (4a) Final EncoderLayer output shape is correct.
 Part 4 seems correct! Assignment complete.


#### **Questions for Part 4**

**Q7: Architecture Check**
What is the purpose of the `self.norm1(x + self.dropout1(attn_output))` operation?

  * (A) It is a residual (skip) connection to help gradients flow and a layer normalization to stabilize training.
  * (B) It is a way to combine the query and key vectors before the final output.
  * (C) It is a non-linear activation function applied to the attention output.
  * (D) It is a bottleneck layer to reduce the dimensionality of the model.


**Q8: Final Value Check**
Using the `encoder_layer` initialized in the verification block (with `seed=42`), run it on the `x_test` input. What is the **mean** of the final `output` tensor?

In [ ]:
# Code for Q8
# (Assumes encoder_layer and x_test are from the verification block)
torch.manual_seed(42)
encoder_layer_q8 = EncoderLayer(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
x_test_q8 = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
output_q8 = encoder_layer_q8(x_test_q8, mask=None)
print(f"Q8 Final Output Mean: {output_q8.mean().item()}")

Q8 Final Output Mean: 0.0




**Q9: Conceptual Check**
What is the role of the `mask` parameter in the `EncoderLayer`?

  * (A) It is *only* used for causal masking in decoder self-attention.
  * (B) It is used to prevent the model from attending to padding tokens in the input sequence.
  * (C) It is used to randomly drop out (mask) tokens to improve generalization.
  * (D) Both (A) and (B) are common uses, but in an *encoder*, (B) is its primary purpose.



**Q10: Total Parameters**
Run the code block below to calculate the total number of parameters in your `EncoderLayer`.

In [ ]:
# Code for Q10
torch.manual_seed(42)
encoder_layer_q10 = EncoderLayer(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
total_params = sum(p.numel() for p in encoder_layer_q10.parameters())
print(f"Q10 Total Parameters: {total_params}")

Q10 Total Parameters: 3152384
